In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("ncr_ride_bookings.csv")

### Handling Duplicates Records

In [ ]:
df.duplicated().any()

In [ ]:
df = df.drop_duplicates(subset="Booking ID")
df["Booking ID"].duplicated().sum()

In [ ]:
df.info()

### Handling Missing Values

In [ ]:
df.isnull().any()

In [ ]:
df = df.fillna("Not Available")

### Creating a Cancellation Column

In [ ]:
def Cancellation_fun(row):
    c = row["Cancelled Rides by Customer"]
    d = row["Cancelled Rides by Driver"]
    if c == "Not Available" and d == "Not Available":
        return "Not Available"
    else:
        if c == "Not Available":
            return d
        elif d == "Not Available":
            return c
        else:
            return c+d
df["Cancellation"] = df.apply(Cancellation_fun,axis = 1)

### Creating a Distance Range Column

In [ ]:
def Dist_Bucket_fun(row):
    r = row["Ride Distance"]
    if r == "Not Available":
        return "Not Available"
    elif r <= 5:
        return "0-5km"
    elif r <=10:
        return "5-10km"
    elif r <=15:
        return "10-15km"
    elif r <=20:
        return "15-20km"
    else:
        return "20+km"
df["Distance_Range"] = df.apply(Dist_Bucket_fun,axis = 1)

### Converting columns into snake_case

In [ ]:
df.columns = df.columns.str.replace(" ","_")

### Formatting Date

In [ ]:
df["Date"] = pd.to_datetime(df["Date"],format="mixed",dayfirst = True, errors ='coerce')

### **FUNNEL ANALYSIS**

In [ ]:
total_bookings = df["Booking_ID"].nunique()

completed = df[df["Booking_Status"] == "Completed"]["Booking_ID"].nunique()

customer_cancel = df[df["Cancelled_Rides_by_Customer"] == 1]["Booking_ID"].nunique()

driver_cancel = df[df["Cancelled_Rides_by_Driver"] == 1]["Booking_ID"].nunique()

cancelled = customer_cancel + driver_cancel

incomplete = df[df["Booking_Status"] == "Incomplete"]["Booking_ID"].nunique()
funnel_df = pd.DataFrame({
    'Stage':["Total_Bookings"  , "Completed" , "Cancelled", "Incomplete"],
    'Count' :[total_bookings , completed ,cancelled ,  incomplete]
},index = None)

funnel_df["% of Total"] = round(
    (funnel_df["Count"] / total_bookings) * 100, 2
)



funnel_df

funnel_df.to_csv("funnel.csv")

In [ ]:
df["Booking_Status"].value_counts()

### Data Quality and Limitations

In [ ]:
Customer_Reason_Missing = round((df[(df["Reason_for_cancelling_by_Customer"]=="Not Available") & (df["Booking_Status"] == "Cancelled by Customer")]["Booking_ID"].nunique() *100)/total_bookings,2)
print(f'Customers_Reason_Missings: {Customer_Reason_Missing}%')

Driver_Reason_Missing = round((df[df["Driver_Cancellation_Reason"] == "Not Available"]["Booking_ID"].nunique() *100)/total_bookings,2)
print(f'Driver_Reason_Missing: {Driver_Reason_Missing}%')

Incomplete_Rides_Missing = round((df[df["Incomplete_Rides_Reason"] == "Not Available"]["Booking_ID"].nunique() *100)/total_bookings,2)
print(f'Incomplete_Rides_Missing: {Incomplete_Rides_Missing}%')

Distance_Range_Missing = round((df[df["Distance_Range"] == "Not Available"]["Booking_ID"].nunique() *100)/total_bookings,2)
print(f'Distance_Range_Missing: {Distance_Range_Missing}%')

In [ ]:
df.to_csv("Uber.csv")